# Optimizing a Prompt with the NeMo Agent Toolkit Optimizer

This notebook builds on `01_nat_basics.ipynb` and shows the smallest useful prompt-optimization loop in the NeMo
Agent Toolkit:

1. Define a workflow that sends `system_prompt + user question` to an LLM.
2. Mark `system_prompt` as optimizable.
3. Score the workflow against a small dataset.
4. Let `nat optimize` rewrite the prompt.
5. Re-run evaluation with the optimized prompt.

No ReAct agent is needed here. No tools are needed here. We only add a tiny notebook-local workflow component because
`nvidia-nat==1.8.0` exposes a built-in `chat_completion` workflow, but that built-in config does not mark
`system_prompt` as an optimizer-aware field. The custom component below is intentionally boring: it exists only to
make `system_prompt` a real `OptimizableField`.

**What we'll optimize**

The task is exact short-answer formatting. The baseline prompt asks for a complete sentence plus an explanation, so
answers like `The capital of France is Paris.` are factually correct but fail exact-match scoring when the reference
answer is just `Paris`. The optimizer's job is to rewrite the prompt so the model returns the short canonical answer
and nothing extra.

## 0. Setup

In [1]:
# Only needed if you did not install requirements.txt into this kernel's environment already.
# - eval: provides EvaluationRun and exact_match evaluation
# - config-optimizer: provides optimize_config (Optuna + genetic-algorithm prompt optimizer)
# %pip install -q "nvidia-nat[langchain,eval,config-optimizer]==1.8.0

In [2]:
import getpass
import os

if not os.environ.get("NVIDIA_API_KEY"):
    os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter your NVIDIA_API_KEY (from build.nvidia.com): ")
else:
    print("NVIDIA_API_KEY already set.")

NVIDIA_API_KEY already set.


## 1. Register a tiny prompt-only workflow

`nat optimize` does not optimize arbitrary YAML strings. It discovers fields whose Python config classes expose
optimizer metadata. The built-in `react_agent.additional_instructions` has that metadata, but a ReAct agent is the
wrong abstraction for this example.

So we register a minimal workflow component named `prompted_chat` in the notebook kernel. Its config has exactly two
important fields:

- `llm_name`: which configured LLM to call
- `system_prompt`: the prompt we want to optimize

The implementation simply sends a system message and a human message to the LLM and returns the LLM's text.

In [3]:
from pydantic import Field

from nat.builder.builder import Builder
from nat.builder.framework_enum import LLMFrameworkEnum
from nat.builder.function_info import FunctionInfo
from nat.cli.register_workflow import register_function
from nat.data_models.component_ref import LLMRef
from nat.data_models.function import FunctionBaseConfig
from nat.data_models.optimizable import OptimizableField
from nat.data_models.optimizable import OptimizableMixin
from nat.data_models.optimizable import SearchSpace


class PromptedChatConfig(FunctionBaseConfig, OptimizableMixin, name="prompted_chat"):
    llm_name: LLMRef = Field(description="LLM to call.")
    system_prompt: str = OptimizableField(
        default="Answer the user.",
        description="System prompt sent to the LLM before the user's question.",
        space=SearchSpace(
            is_prompt=True,
            prompt="Answer the user.",
            prompt_purpose="Improve the model's answer style for the evaluation task.",
        ),
    )


@register_function(config_type=PromptedChatConfig, framework_wrappers=[LLMFrameworkEnum.LANGCHAIN])
async def register_prompted_chat(config: PromptedChatConfig, builder: Builder):
    llm = await builder.get_llm(config.llm_name, wrapper_type=LLMFrameworkEnum.LANGCHAIN)

    async def prompted_chat(question: str) -> str:
        from langchain_core.messages import HumanMessage
        from langchain_core.messages import SystemMessage

        response = await llm.ainvoke([
            SystemMessage(content=config.system_prompt),
            HumanMessage(content=question),
        ])
        return response if isinstance(response, str) else str(response.content)

    yield FunctionInfo.from_fn(prompted_chat, description="Prompt-only chat workflow")


print("Registered workflow type: prompted_chat")

Registered workflow type: prompted_chat


## 2. Create a small exact-answer dataset

`EvaluationRun` sends each `question` to the workflow and compares the workflow's `generated_answer` with the
`answer` field. We use the local `langsmith` / `openevals` `exact_match` evaluator, so the score is `1.0` only when
the generated string exactly equals the reference answer.

That makes the prompt failure visible. Extra words are not "kind of correct" for this metric; they are wrong for the
format we asked the workflow to learn.

In [4]:
%%writefile eval_dataset.json
[
  {"id": "1", "question": "What is the capital of France?", "answer": "Paris"},
  {"id": "2", "question": "What is the chemical symbol for gold?", "answer": "Au"},
  {"id": "3", "question": "How many days are in a week?", "answer": "7"},
  {"id": "4", "question": "Who wrote the book Never Split the Difference?", "answer": "Chris Voss"},
  {"id": "5", "question": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?", "answer": "5 minutes"},
  {"id": "6", "question": "What is the largest planet in our solar system?", "answer": "Jupiter"}
]

Overwriting eval_dataset.json


## 3. Write the workflow + eval + optimizer config

This is the whole workflow now:

```yaml
workflow:
  _type: prompted_chat
  llm_name: answer_llm
  system_prompt: >
    Always answer in one complete sentence and include one short explanation.
  optimizable_params: [system_prompt]
  search_space:
    system_prompt:
      is_prompt: true
      prompt: >
        Always answer in one complete sentence and include one short explanation.
      prompt_purpose: >
        Make the model return exactly the short reference answer and nothing else.
```

`system_prompt` is both the prompt used by the baseline workflow and the prompt seeded into the optimizer. The
`prompt_purpose` tells the GA mutator/crossover functions what a better prompt should accomplish.

In [5]:
%%writefile optimizer_config.yml
functions:
  prompt_mutator:
    _type: prompt_init
    optimizer_llm: optimizer_llm
    optimizer_prompt: >
      Rewrite the instruction so it better achieves the objective below. Keep it short and practical.
      Return only the rewritten instruction text, with no markdown, no code fences, no JSON, and no examples.
      Objective: {system_objective}
    system_objective: >
      Make the model return exactly the short reference answer and nothing else. It should not add explanations,
      labels, markdown, citations, surrounding punctuation, or full sentences when a short answer is sufficient.
  prompt_crossover:
    _type: prompt_recombiner
    optimizer_llm: optimizer_llm
    optimizer_prompt: >
      Merge the strongest parts of two candidate instructions into one concise instruction that best achieves the
      objective below. Return only the merged instruction text, with no markdown, no code fences, no JSON, and no
      examples. Objective: {system_objective}
    system_objective: >
      Make the model return exactly the short reference answer and nothing else. It should not add explanations,
      labels, markdown, citations, surrounding punctuation, or full sentences when a short answer is sufficient.

llms:
  # base_url is driven by optional NVIDIA_BASE_URL; see the README.
  answer_llm:
    _type: nim
    model_name: nvidia/nvidia/nemotron-3-nano-30b-a3b
    max_tokens: 200
    temperature: 0.0
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}
    chat_template_kwargs:
      enable_thinking: false
  optimizer_llm:
    _type: nim
    model_name: nvidia/nvidia/nemotron-3-nano-30b-a3b
    max_tokens: 300
    temperature: 0.7
    api_key: ${NVIDIA_API_KEY}
    base_url: ${NVIDIA_BASE_URL}
    chat_template_kwargs:
      enable_thinking: false

workflow:
  _type: prompted_chat
  llm_name: answer_llm
  system_prompt: >
    Always answer in one complete sentence and include one short explanation.
  optimizable_params: [system_prompt]
  search_space:
    system_prompt:
      is_prompt: true
      prompt: >
        Always answer in one complete sentence and include one short explanation.
      prompt_purpose: >
        Make the model return only the short reference answer and nothing else.

eval:
  general:
    output_dir: ./eval_baseline
    dataset:
      _type: json
      file_path: eval_dataset.json
  evaluators:
    exact_match:
      _type: langsmith
      evaluator: exact_match

optimizer:
  output_path: optimizer_results
  numeric:
    enabled: false
  prompt:
    enabled: true
    prompt_population_init_function: prompt_mutator
    prompt_recombination_function: prompt_crossover
    ga_population_size: 4
    ga_generations: 2
    ga_crossover_rate: 0.7
    ga_mutation_rate: 0.4
    ga_elitism: 1
    ga_selection_method: tournament
    ga_tournament_size: 2
    ga_parallel_evaluations: 2
  reps_per_param_set: 1
  eval_metrics:
    exactness:
      evaluator_name: exact_match
      direction: maximize
      weight: 1.0

Overwriting optimizer_config.yml


### Confirm the optimizer sees `workflow.system_prompt`

This cell is the sanity check. If it does not print `workflow.system_prompt`, the optimizer has nothing to tune.

In [6]:
from nat.plugins.config_optimizer.optimizable_utils import walk_optimizables
from nat.runtime.loader import load_config

config = load_config("optimizer_config.yml")
prompt_spaces = {name: space for name, space in walk_optimizables(config).items() if space.is_prompt}

for name, space in prompt_spaces.items():
    print(f"{name}: {space.prompt!r}")
    print(f"  purpose: {space.prompt_purpose}")

assert "workflow.system_prompt" in prompt_spaces

workflow.system_prompt: 'Always answer in one complete sentence and include one short explanation.\n'
  purpose: Make the model return only the short reference answer and nothing else.



In [7]:
prompt_spaces

{'workflow.system_prompt': SearchSpace(values=None, low=None, high=None, log=False, step=None, is_prompt=True, prompt='Always answer in one complete sentence and include one short explanation.\n', prompt_purpose='Make the model return only the short reference answer and nothing else.\n', prompt_format=None)}

## 4. Baseline: score the intentionally verbose prompt

The baseline prompt asks for sentences and explanations. With exact-match scoring, those extra words should hurt.

In [8]:
from nat.data_models.evaluate_runtime import EvaluationRunConfig
from nat.plugins.eval.cli.evaluate import write_tabular_output
from nat.plugins.eval.runtime.evaluate import EvaluationRun

baseline_run = EvaluationRun(EvaluationRunConfig(config_file="optimizer_config.yml"))
baseline_output = await baseline_run.run_and_evaluate()
write_tabular_output(baseline_output)

/Users/siddharthab/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/contextlib.py:210: UserWarning: WARNING! chat_template_kwargs is not default parameter.
                chat_template_kwargs was transferred to model_kwargs.
                Please confirm that chat_template_kwargs is what you intended.
  return await anext(self.gen)
LangSmith (exact_match): 100%|██████████| 6/6 [00:00<00:00, 1201.46it/s]


=== EVALUATION SUMMARY ===
Workflow Status: COMPLETED (workflow_output.json)
Total Runtime: 1.24s

Per evaluator results:
| Evaluator   |   Avg Score | Output File             |
|-------------|-------------|-------------------------|
| exact_match |           0 | exact_match_output.json |



In [9]:
import json


def summarize_exact_match(eval_path, workflow_path, label):
    with open(eval_path) as f:
        eval_data = json.load(f)
    with open(workflow_path) as f:
        workflow_rows = json.load(f)

    workflow_by_id = {str(row["id"]): row for row in workflow_rows}
    print(f"{label}: exact-match average = {eval_data['average_score']}")
    print()
    print("id | score | expected | generated")
    print("---|-------|----------|----------")
    for item in eval_data["eval_output_items"]:
        row = workflow_by_id[str(item["id"])]
        generated = " ".join(str(row.get("generated_answer", "")).splitlines())
        if len(generated) > 90:
            generated = generated[:87] + "..."
        print(f"{item['id']} | {item['score']} | {row['answer']} | {generated}")

    return {
        "raw_avg": eval_data["average_score"],
        "exact_count": sum(1 for item in eval_data["eval_output_items"] if item["score"] == 1.0),
        "total": len(eval_data["eval_output_items"]),
    }


baseline_accuracy = summarize_exact_match(
    "eval_baseline/exact_match_output.json",
    "eval_baseline/workflow_output.json",
    "Baseline",
)

Baseline: exact-match average = 0.0

id | score | expected | generated
---|-------|----------|----------
1 | False | Paris | The capital of France is Paris, which has been its political and cultural center for ce...
2 | False | Au | The chemical symbol for gold is Au, derived from its Latin name "aurum."
3 | False | 7 | There are seven days in a week, which is a standard unit of time used globally.
4 | False | Chris Voss | Chris Voss wrote the book *Never Split the Difference*, and it is based on his experien...
5 | False | 5 minutes | It would take 5 minutes, because each machine makes one widget in 5 minutes, so 100 mac...
6 | False | Jupiter | Jupiter is the largest planet in our solar system, and it is primarily composed of hydr...


## 5. Run the prompt optimizer

This calls the same optimizer runtime that powers `nat optimize`, but from the notebook kernel so it can see the
notebook-local `prompted_chat` workflow registration.

In [ ]:
from nat.data_models.optimizer import OptimizerRunConfig
from nat.plugins.config_optimizer.optimizer_runtime import optimize_config

await optimize_config(OptimizerRunConfig(config_file="optimizer_config.yml", dataset=None, result_json_path="$"))

## 6. Inspect and apply the optimized prompt

The optimizer writes the winning prompt to `optimizer_results/optimized_prompts.json`. This pinned release does not
write a complete optimized YAML file, so we create one by copying `optimizer_config.yml` and replacing
`workflow.system_prompt`.

In [11]:
import json
from nat.runtime.loader import load_config

config = load_config("optimizer_config.yml")
with open("optimizer_results/optimized_prompts.json") as f:
    optimized_prompts = json.load(f)

optimized_system_prompt = optimized_prompts["workflow.system_prompt"][0]
original_system_prompt = config.workflow.system_prompt

print("Original system_prompt:")
print(" ", original_system_prompt)
print()
print("Optimized system_prompt:")
print(" ", optimized_system_prompt)

Original system_prompt:
  Always answer in one complete sentence and include one short explanation.


Optimized system_prompt:
  Answer with only the short reference answer, no extra words, explanations, punctuation, or sentences.


In [12]:
import pandas as pd

history = pd.read_csv("optimizer_results/ga_history_prompts.csv")
print(history.columns.tolist())
history.tail(10)

['generation', 'index', 'metric::exact_match', 'scalar_fitness']


,generation,index,metric::exact_match,scalar_fitness
0,1,0,0.0,1.000000e-12
1,1,1,1.0,1.000000e+00
2,1,2,1.0,1.000000e+00
3,1,3,1.0,1.000000e+00
4,2,0,1.0,5.000000e-01
5,2,1,1.0,5.000000e-01
6,2,2,1.0,5.000000e-01
7,2,3,1.0,5.000000e-01


In [13]:
import yaml

with open("optimizer_config.yml") as f:
    optimized_config = yaml.safe_load(f)

optimized_config["workflow"]["system_prompt"] = optimized_system_prompt
optimized_config["workflow"]["search_space"]["system_prompt"]["prompt"] = optimized_system_prompt
optimized_config["eval"]["general"]["output_dir"] = "./eval_optimized"

with open("optimizer_results/optimized_config.yml", "w") as f:
    yaml.safe_dump(optimized_config, f, sort_keys=False)

print("Wrote optimizer_results/optimized_config.yml with the optimized system_prompt.")

Wrote optimizer_results/optimized_config.yml with the optimized system_prompt.


## 7. Re-score with the optimized prompt

In [14]:
optimized_run = EvaluationRun(EvaluationRunConfig(config_file="optimizer_results/optimized_config.yml"))
optimized_output = await optimized_run.run_and_evaluate()
write_tabular_output(optimized_output)

/Users/siddharthab/.local/share/uv/python/cpython-3.12.13-macos-aarch64-none/lib/python3.12/contextlib.py:210: UserWarning: WARNING! chat_template_kwargs is not default parameter.
                chat_template_kwargs was transferred to model_kwargs.
                Please confirm that chat_template_kwargs is what you intended.
  return await anext(self.gen)
LangSmith (exact_match): 100%|██████████| 6/6 [00:00<00:00, 1583.95it/s]


=== EVALUATION SUMMARY ===
Workflow Status: COMPLETED (workflow_output.json)
Total Runtime: 1.28s

Per evaluator results:
| Evaluator   |   Avg Score | Output File             |
|-------------|-------------|-------------------------|
| exact_match |           1 | exact_match_output.json |



In [15]:
optimized_accuracy = summarize_exact_match(
    "eval_optimized/exact_match_output.json",
    "eval_optimized/workflow_output.json",
    "Optimized",
)

print()
print(f"Baseline exact answers:  {baseline_accuracy['exact_count']}/{baseline_accuracy['total']}")
print(f"Optimized exact answers: {optimized_accuracy['exact_count']}/{optimized_accuracy['total']}")
print(f"Baseline average:        {baseline_accuracy['raw_avg']}")
print(f"Optimized average:       {optimized_accuracy['raw_avg']}")

Optimized: exact-match average = 1.0

id | score | expected | generated
---|-------|----------|----------
1 | True | Paris | Paris
2 | True | Au | Au
3 | True | 7 | 7
4 | True | Chris Voss | Chris Voss
5 | True | 5 minutes | 5 minutes
6 | True | Jupiter | Jupiter

Baseline exact answers:  0/6
Optimized exact answers: 6/6
Baseline average:        0.0
Optimized average:       1.0


## Wrap-up

This is a prompt-optimization example, not an agent example. The only custom code is a small notebook-local workflow
component that exposes `system_prompt` as an `OptimizableField`; the optimizer, evaluator, dataset loading, LLM
configuration, and result artifacts are all NAT machinery.

The important pattern is:

1. Put the text you want to optimize in a config field.
2. Make that field optimizer-aware with `OptimizableField` and `OptimizableMixin`.
3. Set `optimizable_params` for the field in YAML.
4. Confirm `walk_optimizables(...)` sees it.
5. Run eval -> optimize -> eval.

For real tasks, exact match is only appropriate when the answer really should be canonical and short. For open-ended
responses, use a task-appropriate evaluator instead.